# kgeval walkthrough

This notebook evaluates a small knowledge graph with kgeval, and then a retriever that
reads from it. The steps show scores, findings that point at the exact nodes at fault, a
model's verdicts on what rules cannot check, a criterion of your own, checks for CI,
and a draft schema.

The graph is about Lee Kuan Yew and is shaped like Neo4j LLM Graph Builder output, with
two documents split into five chunks, eleven entities, eleven facts, and one community.
It has a defect planted for every metric, such as a duplicate person, a pronoun
extracted as an entity, a college typed as a person, and one entity for two things named
Raffles, so that every metric has something to find.

## 1. Load the graph

Create the model for sections 6 and 7, then connect to the Neo4j database at
`NEO4J_URI` and create the example graph from `seed.cypher`. When the database requires
authentication, also set `NEO4J_PASSWORD`, and `NEO4J_USERNAME` if it is not `neo4j`.

In [1]:
import os
from pathlib import Path

from neo4j import GraphDatabase

import kgeval

model = kgeval.Model("<model>", cache="verdicts.sqlite")

uri = os.environ["NEO4J_URI"]
password = os.environ.get("NEO4J_PASSWORD")
auth = (os.environ.get("NEO4J_USERNAME", "neo4j"), password) if password else None
driver = GraphDatabase.driver(uri, auth=auth)

with driver.session() as session:
    if session.run("MATCH (n) RETURN count(n)").single(strict=True)[0] == 0:
        session.run(Path("seed.cypher").read_text(encoding="utf-8")).consume()

`Neo4jSource.load()` reads the whole graph into a snapshot, which every metric then evaluates.

In [2]:
graph = kgeval.Neo4jSource(driver).load()
print(f"Loaded {len(graph.nodes)} nodes and {len(graph.relationships)} relationships")

Loaded 19 nodes and 40 relationships


## 2. Evaluate the graph

Without inputs, `kgeval.evaluate` runs the 10 intrinsic metrics, which need nothing
but the graph. Each one scores the share of units that pass, so 1.0 is perfect, and
lists its findings with the element IDs at fault. In a notebook, the report renders
as Markdown.

In [3]:
report = kgeval.evaluate(graph)
report

# kgeval report

- Graph: 19 nodes (11 entities, 7 lexical nodes, 1 community node, 0 other nodes) and 40 relationships (11 facts, 14 provenance relationships)
- Layout: lexical labels Chunk, Document; community labels \_\_Community\_\_; base labels \_\_Entity\_\_; names from name, title, id
- Reproduce with kgeval 0.2.0 on graph `sha256:ad4af2806a9d3de99a1b4789bfc53359f0116cc2ee7f1bbd1401fd4b32261698`

## Scores

### Intrinsic graph metrics (deterministic)

| Metric | Score | Failed | Threshold | Result |
| --- | ---: | ---: | ---: | --- |
| entity_uniqueness | 90.0% | 1 / 10 entities with a name | — |  |
| entity_connectivity | 90.9% | 1 / 11 entities | — |  |
| entity_naming | 81.8% | 2 / 11 entities | — |  |
| entity_provenance | 90.9% | 1 / 11 entities | — |  |
| fact_uniqueness | 90.9% | 1 / 11 facts | — |  |
| fact_provenance | 90.9% | 1 / 11 facts | — |  |
| vocabulary_uniqueness | 85.7% | 2 / 14 vocabulary terms | — |  |
| property_type_consistency | 92.3% | 1 / 13 property values | — |  |
| embedding_consistency | 80.0% | 1 / 5 expected embeddings | — |  |
| extraction_coverage | 80.0% | 1 / 5 chunks | — |  |

## Findings

### entity_uniqueness: 1 of 10 entities with a name failed

- 2 entities share the normalized name 'lee kuan yew': "Lee Kuan Yew" (Person), "lee kuan yew" (Person)
  - nodes: `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:4`, `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:9`

### entity_connectivity: 1 of 11 entities failed

- "Marina Bay Sands" (Landmark) is in no fact with another entity
  - nodes: `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:10`

### entity_naming: 2 of 11 entities failed

- "He" (PERSON) is named by a pronoun
  - nodes: `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:11`
- unnamed (Organization) has no non-blank string in name, title, id
  - nodes: `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:7`

### entity_provenance: 1 of 11 entities failed

- "Marina Bay Sands" (Landmark) is grounded in no chunk
  - nodes: `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:10`

### fact_uniqueness: 1 of 11 facts failed

- 2 facts repeat "Lee Kuan Yew" (Person) -\[:BORN_IN]-> "Singapore" (Location)
  - nodes: `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:4`, `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:5`
  - relationships: `5:f55f160f-fe22-4227-97fd-5c7f875fb3b4:13`, `5:f55f160f-fe22-4227-97fd-5c7f875fb3b4:14`

### fact_provenance: 1 of 11 facts failed

- no chunk grounds both entities of "lee kuan yew" (Person) -\[:FOUNDED]-> "People's Action Party" (Organization)
  - nodes: `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:9`, `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:6`
  - relationships: `5:f55f160f-fe22-4227-97fd-5c7f875fb3b4:17`

### vocabulary_uniqueness: 2 of 14 vocabulary terms failed

- PERSON spells the node type Person differently
  - nodes: `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:11`
- Founded spells the relationship type FOUNDED differently
  - nodes: `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:11`, `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:6`
  - relationships: `5:f55f160f-fe22-4227-97fd-5c7f875fb3b4:18`

### property_type_consistency: 1 of 13 property values failed

- "Kwa Geok Choo" (Person): property born is STRING, while other Person entities hold DATE
  - nodes: `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:8`

### embedding_consistency: 1 of 5 expected embeddings failed

- "Table of contents" (Chunk): no embedding under property embedding, while other Chunk nodes have one
  - nodes: `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:3`

### extraction_coverage: 1 of 5 chunks failed

- no entity was extracted from "Table of contents" (Chunk)
  - nodes: `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:3`


## 3. Act on a finding

Each result holds its counts and its findings. A finding's element IDs select the nodes
at fault, ready to paste into Neo4j Browser.

In [4]:
result = report["entity_uniqueness"]
print(f"{result.metric}: {result.score:.1%} of {result.evaluated} {result.unit} pass\n")
for finding in result.findings:
    print(finding.message)
    print(f"MATCH (n) WHERE elementId(n) IN {list(finding.node_ids)} RETURN n")

entity_uniqueness: 90.0% of 10 entities with a name pass

2 entities share the normalized name 'lee kuan yew': "Lee Kuan Yew" (Person), "lee kuan yew" (Person)
MATCH (n) WHERE elementId(n) IN ['4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:4', '4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:9'] RETURN n


## 4. Measure against references

The intrinsic metrics find defects in what the graph holds. References measure what it
should hold:

| File                                     | Holds                                                      | Used by                      |
| ---------------------------------------- | ---------------------------------------------------------- | ---------------------------- |
| [`schema.yaml`](schema.yaml)             | the graph schema, in neo4j-graphrag's `GraphSchema` format | conformance and completeness |
| [`ground_truth.yaml`](ground_truth.yaml) | every fact `singapore.txt` states                          | precision, recall, and F1    |
| [`questions.yaml`](questions.yaml)       | questions with the supporting facts their answers need     | `competency`                 |

Given these inputs, `kgeval.evaluate` also runs every metric that needs them.

In [5]:
questions = kgeval.load_questions("questions.yaml")
kgeval.evaluate(
    graph,
    schema=kgeval.load_schema("schema.yaml"),
    ground_truth=kgeval.load_ground_truth("ground_truth.yaml"),
    questions=questions,
)

# kgeval report

- Graph: 19 nodes (11 entities, 7 lexical nodes, 1 community node, 0 other nodes) and 40 relationships (11 facts, 14 provenance relationships)
- Layout: lexical labels Chunk, Document; community labels \_\_Community\_\_; base labels \_\_Entity\_\_; names from name, title, id
- Reproduce with kgeval 0.2.0 on graph `sha256:ad4af2806a9d3de99a1b4789bfc53359f0116cc2ee7f1bbd1401fd4b32261698`

## Scores

### Intrinsic graph metrics (deterministic)

| Metric | Score | Failed | Threshold | Result |
| --- | ---: | ---: | ---: | --- |
| entity_uniqueness | 90.0% | 1 / 10 entities with a name | — |  |
| entity_connectivity | 90.9% | 1 / 11 entities | — |  |
| entity_naming | 81.8% | 2 / 11 entities | — |  |
| entity_provenance | 90.9% | 1 / 11 entities | — |  |
| fact_uniqueness | 90.9% | 1 / 11 facts | — |  |
| fact_provenance | 90.9% | 1 / 11 facts | — |  |
| vocabulary_uniqueness | 85.7% | 2 / 14 vocabulary terms | — |  |
| property_type_consistency | 92.3% | 1 / 13 property values | — |  |
| embedding_consistency | 80.0% | 1 / 5 expected embeddings | — |  |
| extraction_coverage | 80.0% | 1 / 5 chunks | — |  |

### Schema graph metrics (deterministic)

| Metric | Score | Failed | Threshold | Result |
| --- | ---: | ---: | ---: | --- |
| entity_conformance | 72.7% | 3 / 11 entities | — |  |
| fact_conformance | 54.5% | 5 / 11 facts | — |  |
| schema_completeness | 76.9% | 3 / 13 declared types and patterns | — |  |
| property_completeness | 80.0% | 3 / 15 declared properties | — |  |

### Reference graph metrics (deterministic)

| Metric | Score | Failed | Threshold | Result |
| --- | ---: | ---: | ---: | --- |
| fact_precision | 60.0% | 2 / 5 distinct facts | — |  |
| fact_recall | 75.0% | 1 / 4 ground truth facts | — |  |
| fact_f1 | 66.7% | 3 / 9 distinct and ground truth facts | — |  |
| entity_precision | 80.0% | 1 / 5 distinct entities | — |  |
| entity_recall | 80.0% | 1 / 5 ground truth entities | — |  |
| entity_f1 | 80.0% | 2 / 10 distinct and ground truth entities | — |  |
| competency | 80.0% | 1 / 5 test questions | — |  |

## Findings

### entity_uniqueness: 1 of 10 entities with a name failed

- 2 entities share the normalized name 'lee kuan yew': "Lee Kuan Yew" (Person), "lee kuan yew" (Person)
  - nodes: `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:4`, `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:9`

### entity_connectivity: 1 of 11 entities failed

- "Marina Bay Sands" (Landmark) is in no fact with another entity
  - nodes: `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:10`

### entity_naming: 2 of 11 entities failed

- "He" (PERSON) is named by a pronoun
  - nodes: `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:11`
- unnamed (Organization) has no non-blank string in name, title, id
  - nodes: `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:7`

### entity_provenance: 1 of 11 entities failed

- "Marina Bay Sands" (Landmark) is grounded in no chunk
  - nodes: `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:10`

### fact_uniqueness: 1 of 11 facts failed

- 2 facts repeat "Lee Kuan Yew" (Person) -\[:BORN_IN]-> "Singapore" (Location)
  - nodes: `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:4`, `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:5`
  - relationships: `5:f55f160f-fe22-4227-97fd-5c7f875fb3b4:13`, `5:f55f160f-fe22-4227-97fd-5c7f875fb3b4:14`

### fact_provenance: 1 of 11 facts failed

- no chunk grounds both entities of "lee kuan yew" (Person) -\[:FOUNDED]-> "People's Action Party" (Organization)
  - nodes: `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:9`, `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:6`
  - relationships: `5:f55f160f-fe22-4227-97fd-5c7f875fb3b4:17`

### vocabulary_uniqueness: 2 of 14 vocabulary terms failed

- PERSON spells the node type Person differently
  - nodes: `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:11`
- Founded spells the relationship type FOUNDED differently
  - nodes: `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:11`, `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:6`
  - relationships: `5:f55f160f-fe22-4227-97fd-5c7f875fb3b4:18`

### property_type_consistency: 1 of 13 property values failed

- "Kwa Geok Choo" (Person): property born is STRING, while other Person entities hold DATE
  - nodes: `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:8`

### embedding_consistency: 1 of 5 expected embeddings failed

- "Table of contents" (Chunk): no embedding under property embedding, while other Chunk nodes have one
  - nodes: `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:3`

### extraction_coverage: 1 of 5 chunks failed

- no entity was extracted from "Table of contents" (Chunk)
  - nodes: `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:3`

### entity_conformance: 3 of 11 entities failed

- "He" (PERSON): undeclared node type PERSON
  - nodes: `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:11`
- unnamed (Organization): KEY constraint on Organization needs property id
  - nodes: `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:7`
- "Kwa Geok Choo" (Person): property born is STRING, expected DATE
  - nodes: `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:8`

### fact_conformance: 5 of 11 facts failed

- "He" (PERSON) -\[:Founded]-> "People's Action Party" (Organization): undeclared relationship type Founded
  - nodes: `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:11`, `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:6`
  - relationships: `5:f55f160f-fe22-4227-97fd-5c7f875fb3b4:18`
- "Kwa Geok Choo" (Person) -\[:KNOWS]-> "Kwa Geok Choo" (Person): undeclared relationship type KNOWS
  - nodes: `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:8`
  - relationships: `5:f55f160f-fe22-4227-97fd-5c7f875fb3b4:20`
- "Lee" (Person) -\[:STUDIED_AT]-> "Fitzwilliam College" (Person): undeclared relationship type STUDIED_AT
  - nodes: `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:15`, `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:16`
  - relationships: `5:f55f160f-fe22-4227-97fd-5c7f875fb3b4:30`
- "Lee Kuan Yew" (Person) -\[:ATTENDED]-> "Raffles" (Organization): undeclared relationship type ATTENDED
  - nodes: `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:4`, `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:17`
  - relationships: `5:f55f160f-fe22-4227-97fd-5c7f875fb3b4:31`
- "Raffles" (Organization) -\[:FOUNDED]-> "Singapore" (Location): pattern (Organization)-\[:FOUNDED]->(Location) is not declared
  - nodes: `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:17`, `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:5`
  - relationships: `5:f55f160f-fe22-4227-97fd-5c7f875fb3b4:32`

### schema_completeness: 3 of 13 declared types and patterns failed

- no entity has the declared node type Award
- no fact has the declared relationship type WON
- no fact realizes the declared pattern (Person)-\[:WON]->(Award)

### property_completeness: 3 of 15 declared properties failed

- "Lee" (Person): declared property born is missing
  - nodes: `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:15`
- "Fitzwilliam College" (Person): declared property born is missing
  - nodes: `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:16`
- unnamed (Organization): declared property id is missing
  - nodes: `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:7`

### fact_precision: 2 of 5 distinct facts failed

- "He" (PERSON) -\[:Founded]-> "People's Action Party" (Organization) is not in the ground truth
  - nodes: `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:11`, `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:6`
  - relationships: `5:f55f160f-fe22-4227-97fd-5c7f875fb3b4:18`
- "Kwa Geok Choo" (Person) -\[:KNOWS]-> "Kwa Geok Choo" (Person) is not in the ground truth
  - nodes: `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:8`
  - relationships: `5:f55f160f-fe22-4227-97fd-5c7f875fb3b4:20`

### fact_recall: 1 of 4 ground truth facts failed

- the graph lacks the ground truth fact "Lee Kuan Yew" -\[:FOUNDED]-> "Lee & Lee"

### entity_precision: 1 of 5 distinct entities failed

- "He" (PERSON) is not in the ground truth
  - nodes: `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:11`

### entity_recall: 1 of 5 ground truth entities failed

- the graph lacks the ground truth entity "Lee & Lee"

### competency: 1 of 5 test questions failed

- 'What did Lee Kuan Yew co-found?': the graph lacks "Lee Kuan Yew" -\[:FOUNDED]-> "Lee & Lee"


## 5. Evaluate retrieval

A retriever is any function that takes a question and returns what it found, as element
IDs or as the nodes, relationships, and paths the Neo4j driver returns. This one returns
the facts around every entity whose name the question contains. `retrieve` asks it every
question and keeps the element IDs; a retriever in another language can write the same
mapping to a JSON file for `load_retrievals` or `kgeval run --retrievals`.

`retrieval_recall` checks that the retriever returned each supporting fact the graph
holds, and `retrieval_precision` that what it returned is needed. `competency` has
already found the one supporting fact the graph lacks, Lee & Lee, so recall does not
blame the retriever for it. Recall does blame it for the question about the founder of
the People's Action Party, whose answer needs two hops while this retriever returns only
one. The entities "He" and "Lee" match every question that contains "he" or "lee", so
their facts lower precision, which shows how defects of the graph cost the retriever.

In [6]:
RETRIEVAL_QUERY = """
MATCH (entity:__Entity__)-[fact]-(other:__Entity__)
WHERE toLower($question) CONTAINS toLower(entity.id)
RETURN entity, fact, other
"""


def retriever(question: str) -> list[object]:
    """Return the facts around every entity whose name the question contains."""
    with driver.session() as session:
        records = session.run(RETRIEVAL_QUERY, question=question)
        return [value for record in records for value in record.values()]


retrievals = kgeval.retrieve(questions, retriever)
kgeval.evaluate(
    graph,
    [kgeval.RetrievalRecall(), kgeval.RetrievalPrecision()],
    questions=questions,
    retrievals=retrievals,
)

# kgeval report

- Graph: 19 nodes (11 entities, 7 lexical nodes, 1 community node, 0 other nodes) and 40 relationships (11 facts, 14 provenance relationships)
- Layout: lexical labels Chunk, Document; community labels \_\_Community\_\_; base labels \_\_Entity\_\_; names from name, title, id
- Reproduce with kgeval 0.2.0 on graph `sha256:ad4af2806a9d3de99a1b4789bfc53359f0116cc2ee7f1bbd1401fd4b32261698`

## Scores

### Retrieval metrics (deterministic)

| Metric | Score | Failed | Threshold | Result |
| --- | ---: | ---: | ---: | --- |
| retrieval_recall | 90.0% | 1 / 6 supporting facts and entities in the graph | — |  |
| retrieval_precision | 40.8% | 48 / 69 retrieved entities, facts, and chunks | — |  |

## Findings

### retrieval_recall: 1 of 6 supporting facts and entities in the graph failed

- "In which country was the founder of the People's Action Party born?": the retrieval lacks "Lee Kuan Yew" -\[:BORN_IN]-> "Singapore"
  - nodes: `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:4`, `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:5`
  - relationships: `5:f55f160f-fe22-4227-97fd-5c7f875fb3b4:13`, `5:f55f160f-fe22-4227-97fd-5c7f875fb3b4:14`

### retrieval_precision: 48 of 69 retrieved entities, facts, and chunks failed

- "Who founded the People's Action Party?": 2 of 7 retrieved entities, facts, and chunks support nothing the answer needs
  - nodes: `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:11`
  - relationships: `5:f55f160f-fe22-4227-97fd-5c7f875fb3b4:18`
- 'Where was Lee Kuan Yew born?': 15 of 19 retrieved entities, facts, and chunks support nothing the answer needs
  - nodes: `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:11`, `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:15`, `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:16`, `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:17`, `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:6`, `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:7`, `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:8`, `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:9`
  - relationships: `5:f55f160f-fe22-4227-97fd-5c7f875fb3b4:15`, `5:f55f160f-fe22-4227-97fd-5c7f875fb3b4:16`, `5:f55f160f-fe22-4227-97fd-5c7f875fb3b4:17`, `5:f55f160f-fe22-4227-97fd-5c7f875fb3b4:18`, `5:f55f160f-fe22-4227-97fd-5c7f875fb3b4:19`, `5:f55f160f-fe22-4227-97fd-5c7f875fb3b4:30`, `5:f55f160f-fe22-4227-97fd-5c7f875fb3b4:31`
- 'What did Lee Kuan Yew co-found?': 12 of 17 retrieved entities, facts, and chunks support nothing the answer needs
  - nodes: `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:15`, `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:16`, `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:17`, `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:5`, `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:7`, `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:8`
  - relationships: `5:f55f160f-fe22-4227-97fd-5c7f875fb3b4:13`, `5:f55f160f-fe22-4227-97fd-5c7f875fb3b4:14`, `5:f55f160f-fe22-4227-97fd-5c7f875fb3b4:16`, `5:f55f160f-fe22-4227-97fd-5c7f875fb3b4:19`, `5:f55f160f-fe22-4227-97fd-5c7f875fb3b4:30`, `5:f55f160f-fe22-4227-97fd-5c7f875fb3b4:31`
- 'When was Lee Kuan Yew born?': 17 of 19 retrieved entities, facts, and chunks support nothing the answer needs
  - nodes: `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:11`, `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:15`, `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:16`, `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:17`, `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:5`, `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:6`, `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:7`, `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:8`
  - relationships: `5:f55f160f-fe22-4227-97fd-5c7f875fb3b4:13`, `5:f55f160f-fe22-4227-97fd-5c7f875fb3b4:14`, `5:f55f160f-fe22-4227-97fd-5c7f875fb3b4:15`, `5:f55f160f-fe22-4227-97fd-5c7f875fb3b4:16`, `5:f55f160f-fe22-4227-97fd-5c7f875fb3b4:17`, `5:f55f160f-fe22-4227-97fd-5c7f875fb3b4:18`, `5:f55f160f-fe22-4227-97fd-5c7f875fb3b4:19`, `5:f55f160f-fe22-4227-97fd-5c7f875fb3b4:30`, `5:f55f160f-fe22-4227-97fd-5c7f875fb3b4:31`
- "In which country was the founder of the People's Action Party born?": 2 of 7 retrieved entities, facts, and chunks support nothing the answer needs
  - nodes: `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:11`
  - relationships: `5:f55f160f-fe22-4227-97fd-5c7f875fb3b4:18`


## 6. Ask a model

Rules cannot tell whether two names stand for one thing, whether a chunk says what a
fact claims, or whether two relationship types mean the same. The six model metrics ask
a model instead. `entity_resolution` finds duplicates named differently,
`entity_disambiguation` entities that stand for several things, `node_type_accuracy`
wrong node types, `fact_accuracy` facts their chunks do not support,
`vocabulary_resolution` synonymous types and property keys, and
`extraction_completeness` facts a chunk states that the graph lacks.

Any model that LiteLLM supports works, from a hosted provider or a local server.
`Model` takes the model as LiteLLM names it, `provider/model`, such as
`openai/gpt-5-mini` or `hosted_vllm/Qwen/Qwen3-8B`, and reads the API key from the
provider's environment variable, such as `OPENAI_API_KEY`. This notebook created the
model in section 1.

The sampled metrics check a reproducible sample of at most `sample_size` units, 100 by
default, so the number of calls stays bounded however large the graph is; this graph is
checked in full. When only part is checked, the score comes with a 95% confidence
interval. The model keeps every verdict in `verdicts.sqlite`, keyed by the model and the
prompt, so a rerun with the same model gives the same scores without calling it again;
delete the file to ask afresh. Each finding carries the model's reason. On the command
line, `kgeval run --model <model> --model-cache verdicts.sqlite` uses the same model and
cache.

In [7]:
kgeval.evaluate(
    graph,
    [
        kgeval.EntityResolution(),
        kgeval.EntityDisambiguation(),
        kgeval.NodeTypeAccuracy(),
        kgeval.FactAccuracy(),
        kgeval.VocabularyResolution(),
        kgeval.ExtractionCompleteness(),
    ],
    model=model,
)

# kgeval report

- Graph: 19 nodes (11 entities, 7 lexical nodes, 1 community node, 0 other nodes) and 40 relationships (11 facts, 14 provenance relationships)
- Layout: lexical labels Chunk, Document; community labels \_\_Community\_\_; base labels \_\_Entity\_\_; names from name, title, id
- Reproduce with kgeval 0.2.0 on graph `sha256:ad4af2806a9d3de99a1b4789bfc53359f0116cc2ee7f1bbd1401fd4b32261698`

## Scores

### Model graph metrics (LLM)

| Metric | Score | Failed | Threshold | Result |
| --- | ---: | ---: | ---: | --- |
| entity_resolution | 80.0% | 2 / 10 entities with a name | — |  |
| entity_disambiguation | 66.7% | 1 / 3 entities with a name in two or more chunks with text | — |  |
| node_type_accuracy | 88.9% | 1 / 9 entities with a name, a node type, and a chunk with text | — |  |
| fact_accuracy | 88.9% | 1 / 9 facts between entities with a name, grounded in a chunk with text | — |  |
| vocabulary_resolution | 91.7% | 1 / 12 distinct vocabulary terms | — |  |
| extraction_completeness | 75.0% | 2 / 8 facts stated in chunks | — |  |

## Findings

### entity_resolution: 2 of 10 entities with a name failed

- "Lee" (Person) duplicates "Lee Kuan Yew" (Person): The excerpt for "Lee" is identical to one of the excerpts for "Lee Kuan Yew" and mentions Lee Kuan Yew by name, indicating they are the same person.
  - nodes: `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:15`, `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:4`
- "lee kuan yew" (Person) duplicates "Lee Kuan Yew" (Person), "Lee" (Person): Duplicate 1 matches exactly by name and excerpts, and duplicate 2's excerpt links 'Lee' to Lee Kuan Yew through shared biographical details, indicating they are the same person.
  - nodes: `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:9`, `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:4`, `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:15`

### entity_disambiguation: 1 of 3 entities with a name in two or more chunks with text failed

- "Raffles" (Organization) stands for more than one thing: The first chunk uses "Raffles" as part of the school name Raffles Institution, while the second uses it to refer to the person who founded Singapore, so they are not the same specific thing.
  - nodes: `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:17`

### node_type_accuracy: 1 of 9 entities with a name, a node type, and a chunk with text failed

- the chunks do not support the node types of "Fitzwilliam College" (Person): The chunk describes Fitzwilliam College as a place where Lee studied law, not as a person.
  - nodes: `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:16`

### fact_accuracy: 1 of 9 facts between entities with a name, grounded in a chunk with text failed

- the chunks do not support "Kwa Geok Choo" (Person) -\[:KNOWS]-> "Kwa Geok Choo" (Person): The chunk only states that Kwa Geok Choo married Lee Kuan Yew and does not mention or imply that she knows herself.
  - nodes: `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:8`
  - relationships: `5:f55f160f-fe22-4227-97fd-5c7f875fb3b4:20`

### vocabulary_resolution: 1 of 12 distinct vocabulary terms failed

- STUDIED_AT means the same as the relationship type ATTENDED: Both connect a person to an educational institution to indicate attendance or study.
  - nodes: `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:15`, `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:16`
  - relationships: `5:f55f160f-fe22-4227-97fd-5c7f875fb3b4:30`

### extraction_completeness: 2 of 8 facts stated in chunks failed

- "Lee Kuan Yew was born in Singapore. He co-founded the Peopl…" (Chunk) states facts the graph lacks: Lee Kuan Yew co-founded the law firm Lee & Lee.
  - nodes: `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:1`
- "Raffles founded modern Singapore in 1819. Lee Kuan Yew beca…" (Chunk) states facts the graph lacks: Lee Kuan Yew became Singapore's first prime minister in 1959.
  - nodes: `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:14`


## 7. Check a criterion

A criterion defines a metric of your own in one sentence, which the model checks on each
entity or fact it applies to, answering yes, no, or not applicable. This one asks for the
day, month, and year of each person's birth, which Kwa Geok Choo's `born` lacks, since it
holds only a year, and which the surname-only "Lee" has no `born` for at all. The college
typed as a person is not a person, so the criterion does not apply to it. In a suite, the
same criterion is an entry with `criterion`, `applies_to: entities`, and
`node_types: [Person]`.

In [8]:
dated_people = kgeval.EntityCriterion(
    name="dated_people",
    criterion="The entity gives the day, month, and year of the person's birth.",
    node_types=frozenset({"Person"}),
)
kgeval.evaluate(graph, [dated_people], model=model)

# kgeval report

- Graph: 19 nodes (11 entities, 7 lexical nodes, 1 community node, 0 other nodes) and 40 relationships (11 facts, 14 provenance relationships)
- Layout: lexical labels Chunk, Document; community labels \_\_Community\_\_; base labels \_\_Entity\_\_; names from name, title, id
- Reproduce with kgeval 0.2.0 on graph `sha256:ad4af2806a9d3de99a1b4789bfc53359f0116cc2ee7f1bbd1401fd4b32261698`

## Scores

### Model graph metrics (LLM)

| Metric | Score | Failed | Threshold | Result |
| --- | ---: | ---: | ---: | --- |
| dated_people | 50.0% | 2 / 4 entities the criterion applies to | — |  |

## Findings

### dated_people: 2 of 4 entities the criterion applies to failed

- "Lee" (Person) does not meet the criterion: The provided text does not state Lee's birth date.
  - nodes: `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:15`
- "Kwa Geok Choo" (Person) does not meet the criterion: The entity only provides the birth year 1920, not the day and month required by the criterion.
  - nodes: `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:8`


## 8. Gate on quality

A threshold turns a metric into a check that fails the report when the metric scores
below it. In a pytest test, `kgeval.assert_passes` fails the test with the report as
its message. On the command line, `kgeval run --threshold entity_uniqueness=0.95` exits
with 1.

In [9]:
check = kgeval.evaluate(graph, [kgeval.EntityUniqueness(threshold=0.95)])
check.passed

False

## 9. Compare with a baseline

A threshold catches a lower score; a baseline catches any new defect. Here the
duplicated `BORN_IN` fact is removed in memory, as a fix in the database would remove
it, and the new run is compared with the first. The fix resolves one finding and adds
none, so nothing regressed, even though a few scores dip with one fact fewer to count.
On the command line, `kgeval compare baseline.json head.json` compares two JSON reports
the same way.

In [10]:
copies = report["fact_uniqueness"].findings[0].relationship_ids[1:]
fixed = kgeval.Graph(
    nodes=graph.nodes,
    relationships=tuple(link for link in graph.relationships if link.id not in copies),
)
kgeval.compare(report, kgeval.evaluate(fixed))

# kgeval comparison

**NO REGRESSION**

- Base: kgeval 0.2.0 on graph `sha256:ad4af2806a9d3de99a1b4789bfc53359f0116cc2ee7f1bbd1401fd4b32261698`, with 11 entities and 11 facts
- Head: kgeval 0.2.0 on graph `sha256:50c2004a9fe82d58615e1a3863b3497288285311d893d45e95578ea7b2be3d45`, with 11 entities and 10 facts

## Summary

Scores: 1 improved, 1 worsened, 8 unchanged, 0 inconclusive

## Scores

| Metric | Base | Head | Change | New | Resolved | Note |
| --- | ---: | ---: | ---: | ---: | ---: | --- |
| entity_uniqueness | 90.0% of 10 | 90.0% of 10 | +0.0 pp | 0 | 0 |  |
| entity_connectivity | 90.9% of 11 | 90.9% of 11 | +0.0 pp | 0 | 0 |  |
| entity_naming | 81.8% of 11 | 81.8% of 11 | +0.0 pp | 0 | 0 |  |
| entity_provenance | 90.9% of 11 | 90.9% of 11 | +0.0 pp | 0 | 0 |  |
| fact_uniqueness | 90.9% of 11 | 100.0% of 10 | +9.1 pp | 0 | 1 |  |
| fact_provenance | 90.9% of 11 | 90.0% of 10 | -0.9 pp | 0 | 0 |  |
| vocabulary_uniqueness | 85.7% of 14 | 85.7% of 14 | +0.0 pp | 0 | 0 |  |
| property_type_consistency | 92.3% of 13 | 92.3% of 13 | +0.0 pp | 0 | 0 |  |
| embedding_consistency | 80.0% of 5 | 80.0% of 5 | +0.0 pp | 0 | 0 |  |
| extraction_coverage | 80.0% of 5 | 80.0% of 5 | +0.0 pp | 0 | 0 |  |

## Resolved findings

### fact_uniqueness: 1 resolved finding

- 2 facts repeat "Lee Kuan Yew" (Person) -\[:BORN_IN]-> "Singapore" (Location)
  - nodes: `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:4`, `4:f55f160f-fe22-4227-97fd-5c7f875fb3b4:5`
  - relationships: `5:f55f160f-fe22-4227-97fd-5c7f875fb3b4:13`, `5:f55f160f-fe22-4227-97fd-5c7f875fb3b4:14`


## 10. Draft a schema from the graph

kgeval drafts the schema the graph follows today, to review and correct instead
of writing it from scratch. The draft copies the graph's defects, such as the `PERSON`
node type, which is what the review should remove.

In [11]:
print(kgeval.infer_schema(kgeval.GraphView(graph)).to_yaml())

node_types:
- label: Person
  properties:
  - {name: born, type: DATE}
  - {name: id, type: STRING}
  additional_properties: false
- label: Organization
  properties:
  - {name: id, type: STRING}
  additional_properties: false
- label: Landmark
  properties:
  - {name: id, type: STRING}
  additional_properties: false
- label: Location
  properties:
  - {name: id, type: STRING}
  additional_properties: false
- label: PERSON
  properties:
  - {name: id, type: STRING}
  additional_properties: false
relationship_types:
- {label: FOUNDED, additional_properties: false}
- {label: BORN_IN, additional_properties: false}
- {label: ATTENDED, additional_properties: false}
- {label: Founded, additional_properties: false}
- {label: KNOWS, additional_properties: false}
- {label: MARRIED, additional_properties: false}
- {label: STUDIED_AT, additional_properties: false}
patterns:
- [Person, FOUNDED, Organization]
- [Person, BORN_IN, Location]
- [Organization, FOUNDED, Location]
- [PERSON, Founded, Orga

## Clean up

Close the connection.

In [12]:
driver.close()